# 02 - Query Infrastructure

## Objective

This notebook loads

- final_disaster_dataset.csv
- india_infrastructure.parquet

For every disaster event, compute

- Nearest Hospital Distance
- Hospital Count within Radius

- Nearest Clinic Distance
- Clinic Count within Radius

- Nearest School Distance
- School Count within Radius

- Nearest Fire Station Distance
- Fire Station Count within Radius

Output

event_infrastructure_summary.csv

In [1]:
from pathlib import Path

import warnings

import numpy as np

import pandas as pd

import geopandas as gpd

from tqdm.auto import tqdm

from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

In [2]:
PROJECT_DIR = Path.cwd().parent

DATASET_DIR = PROJECT_DIR / "datasets"

DISASTER_PATH = (
    DATASET_DIR /
    "final_disaster_dataset.csv"
)

INFRASTRUCTURE_PATH = (
    DATASET_DIR /
    "india_infrastructure.parquet"
)

OUTPUT_PATH = (
    DATASET_DIR /
    "event_infrastructure_summary.csv"
)

SEARCH_RADIUS_KM = 25

In [3]:
events = pd.read_csv(
    DISASTER_PATH
)

infra = gpd.read_parquet(
    INFRASTRUCTURE_PATH
)

print(events.shape)

print(infra.shape)

(2374, 15)
(116701, 5)


In [4]:
events = gpd.GeoDataFrame(

    events,

    geometry=gpd.points_from_xy(

        events.Longitude,

        events.Latitude

    ),

    crs="EPSG:4326"

)

In [5]:
events = events.to_crs("EPSG:3857")

infra = infra.to_crs("EPSG:3857")

# Convert all geometries to representative points
infra["geometry"] = infra.geometry.representative_point()

print(infra.geom_type.value_counts())

Point    116701
Name: count, dtype: int64


In [6]:
hospitals = infra[
    infra.amenity == "hospital"
].copy()

clinics = infra[
    infra.amenity == "clinic"
].copy()

schools = infra[
    infra.amenity == "school"
].copy()

firestations = infra[
    infra.amenity == "fire_station"
].copy()

print(len(hospitals))

print(len(clinics))

print(len(schools))

print(len(firestations))

55605
22553
37815
728


In [7]:
print(infra.geom_type.value_counts())

Point    116701
Name: count, dtype: int64


In [8]:
def build_tree(gdf, name):

    gdf = gdf.copy()

    gdf = gdf[gdf.geometry.notnull()]

    gdf = gdf[gdf.geometry.geom_type == "Point"]

    coords = np.column_stack(
        (
            gdf.geometry.x.to_numpy(),
            gdf.geometry.y.to_numpy()
        )
    )

    tree = cKDTree(coords)

    print(f"{name}: {len(gdf):,} points")

    return tree, gdf

In [9]:
hospital_tree, hospitals = build_tree(
    hospitals,
    "Hospitals"
)

clinic_tree, clinics = build_tree(
    clinics,
    "Clinics"
)

school_tree, schools = build_tree(
    schools,
    "Schools"
)

fire_tree, firestations = build_tree(
    firestations,
    "Fire Stations"
)

print()

print("All KD Trees Built Successfully")

Hospitals: 55,605 points
Clinics: 22,553 points
Schools: 37,815 points
Fire Stations: 728 points

All KD Trees Built Successfully


In [10]:
SEARCH_RADIUS_METERS = (
    SEARCH_RADIUS_KM * 1000
)

In [11]:
def query_tree(tree, x, y):

    distance, index = tree.query(

        [x, y],

        k=1

    )

    neighbours = tree.query_ball_point(

        [x, y],

        SEARCH_RADIUS_METERS

    )

    return (

        distance / 1000,

        len(neighbours)

    )

In [12]:
def query_infrastructure(tree, x, y):

    # nearest infrastructure
    distance, _ = tree.query(
        [x, y],
        k=1
    )

    # all infrastructure within radius
    neighbours = tree.query_ball_point(
        [x, y],
        SEARCH_RADIUS_METERS
    )

    return (
        round(distance / 1000, 3),
        len(neighbours)
    )

In [13]:
nearest_hospital = []
hospital_count = []

nearest_clinic = []
clinic_count = []

nearest_school = []
school_count = []

nearest_fire = []
fire_count = []

In [14]:
for event in tqdm(
    events.itertuples(index=False),
    total=len(events),
    desc="Processing Disaster Events"
):

    x = event.geometry.x
    y = event.geometry.y

    # Hospital
    d, c = query_infrastructure(
        hospital_tree,
        x,
        y
    )

    nearest_hospital.append(d)
    hospital_count.append(c)

    # Clinic
    d, c = query_infrastructure(
        clinic_tree,
        x,
        y
    )

    nearest_clinic.append(d)
    clinic_count.append(c)

    # School
    d, c = query_infrastructure(
        school_tree,
        x,
        y
    )

    nearest_school.append(d)
    school_count.append(c)

    # Fire Station
    d, c = query_infrastructure(
        fire_tree,
        x,
        y
    )

    nearest_fire.append(d)
    fire_count.append(c)

Processing Disaster Events:   0%|          | 0/2374 [00:00<?, ?it/s]

In [15]:
summary = pd.DataFrame({

    "SourceEventID": events["SourceEventID"],

    "NearestHospitalDistanceKm": nearest_hospital,

    "HospitalCount25Km": hospital_count,

    "NearestClinicDistanceKm": nearest_clinic,

    "ClinicCount25Km": clinic_count,

    "NearestSchoolDistanceKm": nearest_school,

    "SchoolCount25Km": school_count,

    "NearestFireStationDistanceKm": nearest_fire,

    "FireStationCount25Km": fire_count

})

summary.head()

,SourceEventID,NearestHospitalDistanceKm,HospitalCount25Km,NearestClinicDistanceKm,ClinicCount25Km,NearestSchoolDistanceKm,SchoolCount25Km,NearestFireStationDistanceKm,FireStationCount25Km
0,18,33.723,0,82.457,0,19.557,1,163.987,0
1,19,25.589,0,77.280,0,26.059,0,162.139,0
2,22,11.797,4,20.231,2,8.987,9,54.765,0
3,39,10.559,9,8.152,3,16.808,24,22.613,1
4,46,9.859,3,34.070,0,9.713,9,36.572,0


In [16]:
summary.to_csv(
    OUTPUT_PATH,
    index=False
)

print()

print("Saved Successfully")

print(OUTPUT_PATH)


Saved Successfully
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\event_infrastructure_summary.csv


In [17]:
verification = pd.read_csv(
    OUTPUT_PATH
)

print(verification.head())

print()

print(verification.describe())

   SourceEventID  NearestHospitalDistanceKm  HospitalCount25Km  \
0             18                     33.723                  0   
1             19                     25.589                  0   
2             22                     11.797                  4   
3             39                     10.559                  9   
4             46                      9.859                  3   

   NearestClinicDistanceKm  ClinicCount25Km  NearestSchoolDistanceKm  \
0                   82.457                0                   19.557   
1                   77.280                0                   26.059   
2                   20.231                2                    8.987   
3                    8.152                3                   16.808   
4                   34.070                0                    9.713   

   SchoolCount25Km  NearestFireStationDistanceKm  FireStationCount25Km  
0                1                       163.987                     0  
1                0      

In [18]:
print()

print("Nearest Hospital")

print(
    verification["NearestHospitalDistanceKm"].describe()
)

print()

print("Hospital Count")

print(
    verification["HospitalCount25Km"].describe()
)

print()

print("Nearest School")

print(
    verification["NearestSchoolDistanceKm"].describe()
)

print()

print("School Count")

print(
    verification["SchoolCount25Km"].describe()
)

print()

print("Nearest Clinic")

print(
    verification["NearestClinicDistanceKm"].describe()
)

print()

print("Clinic Count")

print(
    verification["ClinicCount25Km"].describe()
)

print()

print("Nearest Fire Station")

print(
    verification["NearestFireStationDistanceKm"].describe()
)

print()

print("Fire Station Count")

print(
    verification["FireStationCount25Km"].describe()
)


Nearest Hospital
count    2374.000000
mean       72.373187
std       126.542798
min         0.148000
25%        13.376500
50%        26.746000
75%        47.713000
max       568.010000
Name: NearestHospitalDistanceKm, dtype: float64

Hospital Count
count    2374.000000
mean        6.365628
std        33.062932
min         0.000000
25%         0.000000
50%         0.000000
75%         4.000000
max       739.000000
Name: HospitalCount25Km, dtype: float64

Nearest School
count    2374.000000
mean       79.565080
std       126.464688
min         0.280000
25%        17.299750
50%        34.158500
75%        64.226500
max       573.137000
Name: NearestSchoolDistanceKm, dtype: float64

School Count
count    2374.000000
mean        5.055602
std        45.319434
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max      1178.000000
Name: SchoolCount25Km, dtype: float64

Nearest Clinic
count    2374.000000
mean      100.582197
std       124.451697
min         0

In [19]:
verification.nlargest(
    10,
    "NearestHospitalDistanceKm"
)[
    [
        "SourceEventID",
        "NearestHospitalDistanceKm",
        "HospitalCount25Km"
    ]
]

,SourceEventID,NearestHospitalDistanceKm,HospitalCount25Km
1609,14089,568.010,0
1924,15946,565.651,0
1910,15866,563.438,0
525,4861,560.018,0
1581,13921,548.047,0
2040,16933,545.463,0
418,3953,544.259,0
757,7312,538.844,0
233,2571,534.854,0
714,6924,532.075,0
